In [3]:
import json
import glob
import os

The code below filters the JSON files collected from the Tumblr API in order to only keep posts relating to intersectionality.

In [ ]:
output_file = "intersectionality_matches.json"
seen_ids = set()
first = True

text_fields = ['text', 'body', 'caption', 'description', 'excerpt', 'question', 'answer']
keywords = ['intersectionality', 'intersectional', 'intersect', 'intersection']

# Collect all relevant folders
data_folders = [d for d in os.listdir() if os.path.isdir(d) and d.startswith("data_")]
print(f"📁 Found folders: {len(data_folders)}")

with open(output_file, "w", encoding="utf-8") as out:
    out.write("[\n")  # Start of JSON array

    for folder in sorted(data_folders):
        print(f"\n📂 Processing: {folder}")
        for year in range(2013, 2024):
            monthly_files = sorted(glob.glob(f"{folder}/{year}-*.json"))
            for file in monthly_files:
                print(f"  📄 {file}")
                try:
                    with open(file, "r", encoding="utf-8") as f:
                        data = json.load(f)
                        if not isinstance(data, list):
                            data = [data]
                except Exception as e:
                    print(f"    ⚠️ Skipping {file}: {e}")
                    continue

                for post in data:
                    post_id = post.get("id")
                    if not post_id or post_id in seen_ids:
                        continue
                    seen_ids.add(post_id)

                    # Check text fields
                    found = False
                    for field in text_fields:
                        val = post.get(field)
                        if isinstance(val, str) and any(kw in val.lower() for kw in keywords):
                            found = True
                            break

                    # Check tags
                    if not found and isinstance(post.get('tags'), list):
                        for tag in post['tags']:
                            if isinstance(tag, str) and any(kw in tag.lower() for kw in keywords):
                                found = True
                                break

                    if found:
                        if not first:
                            out.write(",\n")
                        out.write(json.dumps(post, ensure_ascii=False))
                        first = False

    out.write("\n]\n")  # End of JSON array

print(f"\n✅ Done! Matches saved to {output_file}")

